# Examen Final - Caso de Estudio 1: Diagnóstico de Insuficiencia Cardíaca y Ensambles Clínicos
Este notebook simula un caso práctico de examen final que cubre:
1. **Análisis Exploratorio de Datos (AED)** con detección de desbalance de clases.
2. **Preprocesamiento:** Escalado StandardScaler y sobremuestreo sintético **SMOTE**.
3. **Modelado:** Support Vector Machines (SVM) con ajuste de hiperparámetros (GridSearchCV), RandomForest Classifier y un ensamble híbrido **Voting Classifier**.
4. **Evaluación Comparativa:** Accuracy, Precision, Recall, F1-Score, Curva ROC y Curva Precision-Recall (AUC-PR).
5. **Banco de Preguntas Teórico-Prácticas de Examen** al final del notebook.


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_curve, auc, precision_recall_curve, confusion_matrix, ConfusionMatrixDisplay
)
from imblearn.over_sampling import SMOTE

import warnings
warnings.filterwarnings('ignore')


## 1. Análisis Exploratorio de Datos (AED)


In [ ]:
# Carga del dataset de falla cardíaca
# Nota: Asegúrate de tener 'heart_dataset.csv' en la carpeta Sem_9
data = pd.read_csv('Sem_9/heart_dataset.csv')
print("Dimensiones del dataset:", data.shape)
data.head()


In [ ]:
# Estadísticas descriptivas básicas
data.describe()


In [ ]:
# --- AED: DISTRIBUCIÓN DE VARIABLE OBJETIVO Y TABLAS DE CONTINGENCIA APILADAS AL 100% ---
target_col = 'DEATH_EVENT'
conteo_clases = data[target_col].value_counts()
print("Conteo de clases:\n", conteo_clases)

# 1. Gráfico de barras simple para desbalance
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
sns.countplot(x=target_col, data=data, palette='Set2', ax=axes[0])
axes[0].set_title('Distribución de DEATH_EVENT')

# 2. Gráfico apilado al 100% (crosstab) para smoking vs DEATH_EVENT
# Muestra la proporción relativa dentro de cada categoría
tab_smoking = pd.crosstab(data['smoking'], data[target_col])
tab_smoking.div(tab_smoking.sum(1), axis=0).plot(kind='bar', stacked=True, ax=axes[1], color=['#4F81BD', '#C0504D'])
axes[1].set_title('Fumadores vs DEATH_EVENT (100% Stacked)')
axes[1].set_ylabel('Proporción')

# 3. Gráfico apilado al 100% para high_blood_pressure vs DEATH_EVENT
tab_hbp = pd.crosstab(data['high_blood_pressure'], data[target_col])
tab_hbp.div(tab_hbp.sum(1), axis=0).plot(kind='bar', stacked=True, ax=axes[2], color=['#4F81BD', '#C0504D'])
axes[2].set_title('Presión Alta vs DEATH_EVENT (100% Stacked)')
axes[2].set_ylabel('Proporción')

plt.tight_layout()
plt.show()


In [ ]:
# Mapa de calor de correlación para identificar variables de influencia
plt.figure(figsize=(10, 8))
sns.heatmap(data.corr(), annot=True, cmap='coolwarm', fmt='.2f', linewidths=0.5)
plt.title('Matriz de Correlación de Variables')
plt.show()


## 2. Preprocesamiento de Datos


In [ ]:
# Separar características (X) y objetivo (y)
X = data.drop(target_col, axis=1)
y = data[target_col]

# Dividir el conjunto de datos de manera estratificada (70% Train, 30% Test)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print("Distribución de clases en Train:", y_train.value_counts().to_dict())
print("Distribución de clases en Test:", y_test.value_counts().to_dict())


In [ ]:
# Escalado de variables utilizando StandardScaler
# ¿Por qué escalar? SVM se basa en calcular distancias geométricas para maximizar el margen.
# Si no escalamos, variables con magnitudes grandes (como 'creatinine_phosphokinase' o 'platelets')
# dominarán completamente a variables con rangos pequeños (como 'age' o 'anaemia').
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


In [ ]:
# Aplicar SMOTE solo al conjunto de entrenamiento para balancear las clases
smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train_scaled, y_train)

print("Distribución en entrenamiento tras SMOTE:", y_train_res.value_counts().to_dict())


## 3. Modelado


### Modelo A: Support Vector Machine (SVM) con GridSearchCV
Ajustaremos los parámetros C, gamma y kernel usando validación cruzada (GridSearchCV).


In [ ]:
param_grid = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 'auto', 0.01, 0.1],
    'kernel': ['rbf', 'linear']
}

grid_svm = GridSearchCV(SVC(probability=True, random_state=42), param_grid, cv=5, scoring='f1', n_jobs=-1)
grid_svm.fit(X_train_res, y_train_res)

print("Mejores parámetros para SVM:", grid_svm.best_params_)
svm_model = grid_svm.best_estimator_


### **El Framework Universal de Optimización de Hiperparámetros en Scikit-Learn**

En Scikit-Learn existe una estructura **estándar y universal** para buscar hiperparámetros de cualquier modelo usando `GridSearchCV` (búsqueda exhaustiva por grilla) o `RandomizedSearchCV` (búsqueda aleatoria, preferible si la grilla es muy grande).

#### **Estructura Universal:**
```python
from sklearn.model_selection import GridSearchCV

# 1. Definir el estimador base
modelo = AlgoritmoClassifier()

# 2. Definir el diccionario de parámetros a probar (varía por modelo)
param_grid = { 'nombre_parametro': [valores] }

# 3. Configurar GridSearchCV
grid = GridSearchCV(estimator=modelo, param_grid=param_grid, cv=5, scoring='f1', n_jobs=-1)

# 4. Entrenar y obtener el mejor modelo
grid.fit(X_train, y_train)
mejor_modelo = grid.best_estimator_
```

--- 
A continuación, aplicamos esta metodología para optimizar tanto **Random Forest** como **AdaBoost Classifier** (mostrando cómo varían sus grillas de parámetros correspondientes).

In [ ]:
# --- OPTIMIZACIÓN DE RANDOM FOREST (BAGGING) ---
# Parámetros clave de Random Forest:
# - n_estimators: Número de árboles en el bosque.
# - max_depth: Altura máxima de cada árbol.
# - min_samples_split: Muestras mínimas requeridas para dividir un nodo interno.
rf_base = RandomForestClassifier(random_state=42)

param_grid_rf = {
    'n_estimators': [50, 100, 200],
    'max_depth': [5, 10, None],
    'min_samples_split': [2, 5, 10]
}

grid_rf = GridSearchCV(estimator=rf_base, param_grid=param_grid_rf, cv=5, scoring='f1', n_jobs=-1)
grid_rf.fit(X_train_res, y_train_res)

print("Mejores parámetros para Random Forest:", grid_rf.best_params_)
rf_model = grid_rf.best_estimator_

### **Modelo B.2: AdaBoost Classifier (Boosting)**
Optimizamos AdaBoost. Ojo: Dado que AdaBoost entrena un modelo base de forma secuencial (por defecto un `DecisionTreeClassifier(max_depth=1)`), si queremos cambiar parámetros del modelo interno, en la grilla usamos la sintaxis con doble guión bajo: `estimator__<parametro>` (o `base_estimator__<parametro>` en versiones antiguas de sklearn).

In [ ]:
# --- OPTIMIZACIÓN DE ADABOOST (BOOSTING) ---
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

# Definimos el estimador base (árbol de decisión)
base_tree = DecisionTreeClassifier(random_state=42)
ada_base = AdaBoostClassifier(estimator=base_tree, random_state=42)

# Parámetros clave de AdaBoost:
# - n_estimators: Número de clasificadores débiles en secuencia.
# - learning_rate: Qué tanto encoge la contribución de cada clasificador.
# - estimator__max_depth: Profundidad del modelo de árbol base.
param_grid_ada = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.01, 0.1, 0.5],
    'estimator__max_depth': [1, 2, 3] # Nótese el prefijo 'estimator__' para acceder al DecisionTree base
}

grid_ada = GridSearchCV(estimator=ada_base, param_grid=param_grid_ada, cv=5, scoring='f1', n_jobs=-1)
grid_ada.fit(X_train_res, y_train_res)

print("Mejores parámetros para AdaBoost:", grid_ada.best_params_)
ada_model = grid_ada.best_estimator_

### Modelo C: Voting Classifier (Ensamble de Votación)
Combinación híbrida (Soft Voting) usando Regresión Logística, Random Forest y SVM.


In [ ]:
lr_base = LogisticRegression(random_state=42)
rf_base = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42)
svm_base = SVC(C=1, kernel='rbf', gamma='scale', probability=True, random_state=42)

voting_model = VotingClassifier(
    estimators=[('lr', lr_base), ('rf', rf_base), ('svm', svm_base)],
    voting='soft'
)
voting_model.fit(X_train_res, y_train_res)


## 4. Evaluación Comparativa


In [ ]:
# Generar predicciones para los modelos
y_pred_svm = svm_model.predict(X_test_scaled)
y_prob_svm = svm_model.predict_proba(X_test_scaled)[:, 1]

y_pred_rf = rf_model.predict(X_test_scaled)
y_prob_rf = rf_model.predict_proba(X_test_scaled)[:, 1]

y_pred_ada = ada_model.predict(X_test_scaled)
y_prob_ada = ada_model.predict_proba(X_test_scaled)[:, 1]

y_pred_voting = voting_model.predict(X_test_scaled)
y_prob_voting = voting_model.predict_proba(X_test_scaled)[:, 1]


In [ ]:
# Tabla Comparativa de Métricas básicas en Test
resultados = []
for nombre, y_pred, y_prob in [("SVM Optimo", y_pred_svm, y_prob_svm), 
                               ("Random Forest Optimo", y_pred_rf, y_prob_rf), 
                               ("AdaBoost Optimo", y_pred_ada, y_prob_ada), 
                               ("Ensamble Votación (Soft)", y_pred_voting, y_prob_voting)]:
    resultados.append({
        "Modelo": nombre,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred),
        "Recall": recall_score(y_test, y_pred),
        "F1-Score": f1_score(y_test, y_pred)
    })

df_res = pd.DataFrame(resultados)
df_res


In [ ]:
# Graficar las curvas ROC de los modelos
fpr_svm, tpr_svm, _ = roc_curve(y_test, y_prob_svm)
fpr_rf, tpr_rf, _ = roc_curve(y_test, y_prob_rf)
fpr_ada, tpr_ada, _ = roc_curve(y_test, y_prob_ada)
fpr_voting, tpr_voting, _ = roc_curve(y_test, y_prob_voting)

plt.figure(figsize=(8, 6))
plt.plot(fpr_svm, tpr_svm, label=f'SVM (AUC = {auc(fpr_svm, tpr_svm):.3f})')
plt.plot(fpr_rf, tpr_rf, label=f'Random Forest (AUC = {auc(fpr_rf, tpr_rf):.3f})')
plt.plot(fpr_ada, tpr_ada, label=f'AdaBoost (AUC = {auc(fpr_ada, tpr_ada):.3f})')
plt.plot(fpr_voting, tpr_voting, label=f'Voting Classifier (AUC = {auc(fpr_voting, tpr_voting):.3f})')
plt.plot([0, 1], [0, 1], 'k--')
plt.title('Curvas ROC Comparativas - Diagnóstico Cardíaco')
plt.xlabel('Tasa de Falsos Positivos (FPR)')
plt.ylabel('Tasa de Verdaderos Positivos (TPR)')
plt.legend()
plt.show()


In [ ]:
# Graficar las curvas Precision-Recall (PR) de los modelos
prec_svm, rec_svm, _ = precision_recall_curve(y_test, y_prob_svm)
prec_rf, rec_rf, _ = precision_recall_curve(y_test, y_prob_rf)
prec_ada, rec_ada, _ = precision_recall_curve(y_test, y_prob_ada)
prec_voting, rec_voting, _ = precision_recall_curve(y_test, y_prob_voting)

plt.figure(figsize=(8, 6))
plt.plot(rec_svm, prec_svm, label=f'SVM')
plt.plot(rec_rf, prec_rf, label=f'Random Forest')
plt.plot(rec_ada, prec_ada, label=f'AdaBoost')
plt.plot(rec_voting, prec_voting, label=f'Voting Classifier')
plt.title('Curvas Precision-Recall Comparativas')
plt.xlabel('Recall (Sensibilidad)')
plt.ylabel('Precision (Exactitud de Alarma)')
plt.legend()
plt.show()


In [ ]:
# Matrices de Confusión
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred_svm, ax=axes[0], cmap='Blues', colorbar=False)
axes[0].set_title('Matriz de Confusión: SVM')

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_rf, ax=axes[1], cmap='Blues', colorbar=False)
axes[1].set_title('Matriz de Confusión: Random Forest')

ConfusionMatrixDisplay.from_predictions(y_test, y_pred_voting, ax=axes[2], cmap='Blues', colorbar=False)
axes[2].set_title('Matriz de Confusión: Voting')
plt.show()


## 5. Banco de Preguntas y Respuestas Teórico-Prácticas (Nivel Examen)

### **Pregunta 1 (Análisis de Hiperparámetros):**
**Enunciado:** Si observamos que el modelo SVM de Falla Cardíaca arroja un F1-Score muy alto en Train (cercano a 1.0) pero el F1-Score en Test cae abruptamente, ¿cuál de las siguientes combinaciones de hiperparámetros recomendaría ajustar para combatir este fenómeno? Justifique detalladamente.
*   a) Aumentar C y aumentar gamma.
*   b) Disminuir C y disminuir gamma.
*   c) Cambiar el kernel de RBF a polinomial de grado alto.

**Respuesta Correcta y Justificación:**
La respuesta correcta es **b) Disminuir C y disminuir gamma**.
El fenómeno descrito es un claro caso de **Sobreajuste (Overfitting)**. 
- Al **disminuir C**, el algoritmo se vuelve más tolerante a errores en el set de entrenamiento, lo que ensancha el margen de separación y suaviza la frontera de decisión, mejorando la generalización.
- Al **disminuir gamma**, se incrementa el radio de influencia de cada vector de soporte individual, lo que provoca que la curva límite sea más suave y menos propensa a ajustarse al ruido de observaciones específicas.

---

### **Pregunta 2 (Métricas y Negocio):**
**Enunciado:** En este escenario de diagnóstico de insuficiencia cardíaca, la variable positiva (1) representa el fallecimiento del paciente. Explique qué consecuencias tendría un **Falso Negativo (FN)** y un **Falso Positivo (FP)** desde el punto de vista médico y de gestión de salud, y concluya cuál de las siguientes métricas (Precision o Recall) debe priorizar el sistema.

**Respuesta Correcta y Justificación:**
- **Falso Negativo (FN):** Significa que el modelo clasifica al paciente como fuera de peligro (0), pero en realidad tiene un riesgo inminente de fallecer (1). La consecuencia es crítica: el paciente no recibirá cuidados intensivos, monitoreo o tratamiento oportuno, lo que puede resultar en su muerte.
- **Falso Positivo (FP):** Significa que el modelo clasifica al paciente en riesgo alto (1), pero en realidad se encuentra fuera de peligro (0). La consecuencia es que se le asignarán camas, medicamentos o exámenes médicos costosos e innecesarios, lo que genera sobrecostos operativos y estrés innecesario al paciente.
- **Conclusión:** En aplicaciones médicas donde la vida está en juego, **se debe priorizar el RECALL** para minimizar a toda costa los Falsos Negativos. Es preferible incurrir en falsas alarmas (Falsos Positivos) que dar de alta erróneamente a un paciente en estado crítico.

---

### **Pregunta 3 (Comparación de Curvas):**
**Enunciado:** Justifique por qué la curva **Precision-Recall (PR)** es una herramienta de evaluación más informativa y honesta que la curva **ROC** para este conjunto de datos médico si la tasa de muerte fuera del 1% del total.

**Respuesta Correcta y Justificación:**
La curva ROC dibuja la Tasa de Verdaderos Positivos frente a la Tasa de Falsos Positivos ($FPR = \frac{FP}{FP+VN}$). Si la clase mayoritaria (pacientes sanos) es masiva, el número de Verdaderos Negativos (VN) en el denominador del FPR será gigantesco. Esto hace que el FPR permanezca artificialmente cercano a cero, estirando la curva ROC hacia la esquina superior izquierda y simulando un rendimiento excelente del modelo.
La curva Precision-Recall no incluye a los Verdaderos Negativos en ninguna de sus fórmulas. Se enfoca estrictamente en los Verdaderos Positivos, Falsos Positivos y Falsos Negativos, lo que la hace altamente sensible al desbalance y revela con precisión cuántos pacientes realmente en riesgo estamos detectando sin generar excesivas falsas alarmas.

---

### **Pregunta 4 (Interpretación de Ensambles):**
**Enunciado:** Explique la diferencia conceptual entre la opción de votación **Soft Voting** y **Hard Voting** dentro del `VotingClassifier` utilizado. ¿Por qué se prefiere generalmente Soft Voting en producción?

**Respuesta Correcta y Justificación:**
- **Hard Voting:** Realiza un conteo simple de votos de clase. Si el clasificador 1 dice 'Clase 0', el clasificador 2 dice 'Clase 0', y el clasificador 3 dice 'Clase 1', el ensamble predice 'Clase 0' por mayoría de 2 a 1. Ignora la confianza de cada predicción.
- **Soft Voting:** Promedia los vectores de probabilidad devueltos por cada modelo. Si un clasificador predice la Clase 1 con $99\%$ de probabilidad, y los otros dos predicen la Clase 0 de forma insegura con $51\%$ de probabilidad, el promedio de probabilidades favorecerá a la Clase 1. Se prefiere en producción porque da mayor peso a las decisiones tomadas con alto nivel de certeza por parte de los modelos individuales.

---

### **Pregunta 5 (Pregunta Hipotética sobre SMOTE):**
**Enunciado:** Imagine que decide aplicar SMOTE a todo el conjunto de datos original **antes** de realizar la partición `train_test_split`. Describa el impacto que tendrá esta decisión sobre la evaluación de su modelo y cómo se le conoce a este error metodológico.

**Respuesta Correcta y Justificación:**
A este error se le conoce como **Data Leakage (Filtración de Datos)**.
Si aplicamos SMOTE a todo el dataset antes de dividirlo, el algoritmo generará observaciones sintéticas de la clase minoritaria basándose en la distancia entre observaciones que terminarán tanto en el conjunto de entrenamiento como en el de prueba. Esto significa que el conjunto de prueba ya no será un set independiente y desconocido para el modelo, sino que contendrá información filtrada del conjunto de entrenamiento. La métrica en Test será artificialmente perfecta, pero el modelo fallará catastróficamente al implementarse con datos nuevos y reales en producción.
